This notebook is the gallery page [Iterated rippled noise](https://choyun1.github.io/sonore/gallery/irn.html), cell for cell. Run it from top to bottom: the first cell installs sonore, and the recordings come from the sonore repository.

In [ ]:
%pip install --quiet git+https://github.com/choyun1/sonore

# Iterated rippled noise

Add noise to a copy of itself delayed by $d$, and do it again to the result, many times over.
What comes out is still noise, with no tones in it, yet it has a pitch at $1/d$ (Yost, 1996).
It is a classic stimulus for pitch because its strength can be turned up step by step.

- [How it is made](https://choyun1.github.io/sonore/gallery/irn.html#h-how-it-is-made): the delay-and-add network, and what it does to the
  spectrum and the autocorrelation.
- [Pitch from delay](https://choyun1.github.io/sonore/gallery/irn.html#h-pitch-from-delay): an 8 ms delay, sixteen times over, and its pitch at
  125 Hz.
- [Fewer iterations](https://choyun1.github.io/sonore/gallery/irn.html#h-fewer-iterations): one and four iterations, from a faint pitch to a clear
  one.
- [Subtracting instead](https://choyun1.github.io/sonore/gallery/irn.html#h-subtracting-instead): a negative gain, whose pitch is ambiguous.

## How it is made

`so.iterated_ripple_noise` starts from Gaussian noise $x$ and repeats $y \leftarrow y + g\,y(t -
d)$ for the given number of iterations $n$ (the "add-same" network; `network="add-original"`
adds the delayed copy to the original noise instead). In frequency the delay-and-add is a comb
filter, so the whole network is

$$H(f) = \left(1 + g\,e^{-i 2\pi f d}\right)^n,$$

whose peaks, with $g = 1$, sit at multiples of $1/d$ and grow sharper with each iteration. In
time, the noise becomes partly correlated with itself $d$ later: its autocorrelation has a peak
at lag $d$, and its height, which grows with $n$, goes with how strong the pitch is. sonore
computes the network exactly in the frequency domain, so the delay need not be a whole number
of samples, and drops a warm-up stretch so that the noise is stationary from the start.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np

import sonore as so

plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "figure.dpi": 100})


def fetch(path):
    """A file from the sonore repository, by its path there: the local copy when this runs from
    the repository root, otherwise downloaded from GitHub to the same relative path."""
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/choyun1/sonore/main/" + path, path)
    return path


def finish(snd):
    """How every sound in the gallery is played: 5 ms ramps, RMS 0.1, peak at most 0.95."""
    snd = snd.ramp(5e-3).normalize(rms=0.1)
    return snd.normalize(peak=0.95) if snd.peak > 0.95 else snd

In [ ]:
FS = 44100


def show(snd):
    """so.overview: waveform, spectrum, spectrogram (50 ms windows) and modulation spectrum.
    Returns the figure and the panels the playhead follows."""
    fig = so.overview(snd, win_dur=50e-3, figsize=(10, 6.2), fmax=4000)
    return fig, [ax for ax in fig.axes if ax.get_title() in ("Waveform", "Spectrogram")]

## Pitch from delay

**Iterated rippled noise**

Noise added to an 8 ms delayed copy of itself, sixteen times over. A 125 Hz pitch rises out
of the hiss; the spectrum ripples at multiples of 125 Hz and the modulation spectrum peaks at
8 cycles/kHz, the delay in milliseconds, and at zero rate, since the ripple does not move. How
this ripple differs from a [moving ripple](https://choyun1.github.io/sonore/gallery/ripples.html#d-01), and why only this one has a
pitch, is on [Spectrotemporal ripples](https://choyun1.github.io/sonore/gallery/ripples.html#h-ripples-and-rippled-noise).

In [ ]:
sound = finish(so.iterated_ripple_noise(2, FS, delay=8e-3, iterations=16, rng=0))
fig, playhead = show(sound)
sound

## Fewer iterations

The same delay and the same noise, with fewer iterations. The ripple in the spectrum is
shallower and the autocorrelation peak at 8 ms lower, and the pitch is weaker.

**One iteration**

One iteration: noise plus a single delayed copy. The spectrum is a gentle cosine ripple, and
the pitch is faint, a coloring of the hiss rather than a note.

In [ ]:
sound = finish(so.iterated_ripple_noise(2, FS, delay=8e-3, iterations=1, rng=0))
fig, playhead = show(sound)
sound

**Four iterations**

Four iterations: the peaks at multiples of 125 Hz are sharper, and the pitch is clear.

In [ ]:
sound = finish(so.iterated_ripple_noise(2, FS, delay=8e-3, iterations=4, rng=0))
fig, playhead = show(sound)
sound

**Autocorrelation and iterations**

The normalized autocorrelation of each sound over its first 30 ms. The peak at 8 ms grows
with the number of iterations (0.47, 0.75 and 0.88), and so do the peaks at its multiples.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 4.8), sharex=True, layout="constrained")
for ax, n, color in zip(axes, (1, 4, 16), ("tab:blue", "tab:orange", "tab:green"), strict=True):
    x = so.iterated_ripple_noise(2, FS, delay=8e-3, iterations=n, rng=0).data[:, 0]
    X = np.fft.rfft(x, 2 * len(x))
    r = np.fft.irfft(np.abs(X) ** 2)[: int(0.030 * FS)]
    ax.plot(1e3 * np.arange(len(r)) / FS, r / r[0], color=color, lw=1)
    ax.set(ylim=(-0.3, 1.05), ylabel="Autocorr.", title=f"{n} iteration{'s' * (n > 1)}")
    ax.grid(ls=":")
axes[-1].set(xlabel="Lag [ms]", xlim=(0, 30))

## Subtracting instead

**Negative gain**

Sixteen iterations with gain $g = -1$: the delayed copy is subtracted. The spectral peaks
move to odd multiples of 62.5 Hz, halfway between the ones above, and the autocorrelation at
8 ms turns negative, with the first positive peak at 16 ms. The pitch is ambiguous: neither
simply 125 Hz nor an octave below (Yost, 1996).

In [ ]:
sound = finish(so.iterated_ripple_noise(2, FS, delay=8e-3, gain=-1, iterations=16, rng=0))
fig, playhead = show(sound)
sound

## References

- Yost (1996). Pitch of iterated rippled noise. *J. Acoust. Soc. Am.* 100(1), 511–518.
  [JASA (PDF)](https://pubs.aip.org/asa/jasa/article-pdf/100/1/511/11401642/511_1_online.pdf).
  [`waveforms.iterated_ripple_noise`](https://github.com/choyun1/sonore/blob/main/src/sonore/sources/waveforms.py#L591)